In [ ]:
import os
import geopandas as gpd
import pandas as pd
import osmnx as ox

# ----------------------------
# CONFIGURAÇÕES
# ----------------------------
OUT_GPKG = r"D:\Arq-Azzoni\UrbanSprawl\Bases_dados\Areas_verdes\Equipamentos_verdes_SP\equipamentos_verdes_sp.gpkg"
os.makedirs(os.path.dirname(OUT_GPKG), exist_ok=True)

SHAPE_MUNICIPIOS = r"D:\Arq-Azzoni\UrbanSprawl\Bases_dados\Shapes\BR_Municipios_2024\BR_Municipios_2024.shp" # ajuste o nome do seu arquivo
OUT_LAYER = "equipamentos_verdes_sp"

# Tags OSM
tags_parques = {"leisure": ["park", "garden"], "landuse": ["recreation_ground"]}
tags_pracas  = {"place": "square"}
tags_ped     = {"highway": "pedestrian", "area": "yes"}


# ----------------------------
# 1) CARREGAR MUNICÍPIOS
# ----------------------------
munis = gpd.read_file(SHAPE_MUNICIPIOS)

munis = munis[munis['SIGLA_UF'] == 'SP']  # filtrar só SP

# Garantir CRS em WGS84
munis = munis.to_crs(4326)


assert "CD_MUN" in munis.columns
assert "NM_MUN" in munis.columns

print(f"Total de municípios a processar: {len(munis)}")

# Lista de erros
erros = []

# ----------------------------
# FUNÇÃO AUXILIAR
# ----------------------------
def clean_clip(gdf, muni_geom):
    if gdf is None or gdf.empty:
        return gpd.GeoDataFrame(geometry=[], crs=4326)

    gdf = gdf.set_crs(4326, allow_override=True)
    gdf = gpd.clip(gdf, muni_geom)
    gdf = gdf[gdf.geometry.type.isin(["Polygon", "MultiPolygon"])]

    for col in ["name", "leisure"]:
        if col not in gdf.columns:
            gdf[col] = None

    return gdf[["geometry", "name", "leisure"]]

# ----------------------------
# LOOP MUNICÍPIO A MUNICÍPIO
# ----------------------------
primeiro = True
total_registros = 0

for idx, row in munis.iterrows():
    cd_mun = row["CD_MUN"]
    nm_mun = row["NM_MUN"]
    geom_mun = row.geometry

    print(f"Processando {nm_mun} ({cd_mun})...")

    try:
        parks   = ox.features_from_polygon(geom_mun, tags_parques)
        squares = ox.features_from_polygon(geom_mun, tags_pracas)
        ped     = ox.features_from_polygon(geom_mun, tags_ped)

        if not ped.empty:
            ped = ped[ped.get("name", "").astype(str).str.startswith("Praça")]

        parks_poly       = clean_clip(parks, geom_mun)
        squares_poly     = clean_clip(squares, geom_mun)
        ped_pracas_poly  = clean_clip(ped, geom_mun)

        pracas_poly = gpd.GeoDataFrame(
            pd.concat([squares_poly, ped_pracas_poly], ignore_index=True),
            crs=4326
        )

        parks_out = parks_poly.copy()
        parks_out["equipamento"] = "parque"

        pracas_out = pracas_poly.copy()
        pracas_out["equipamento"] = "praça"

        muni_equip = gpd.GeoDataFrame(
            pd.concat([parks_out, pracas_out], ignore_index=True),
            crs=4326
        )

        if muni_equip.empty:
            print(f"  → Nenhum equipamento encontrado.")
            continue

        muni_equip["CD_MUN"] = cd_mun
        muni_equip["NM_MUN"] = nm_mun

        if primeiro:
            muni_equip.to_file(
                OUT_GPKG, layer=OUT_LAYER, driver="GPKG", mode="w"
            )
            primeiro = False
        else:
            muni_equip.to_file(
                OUT_GPKG, layer=OUT_LAYER, driver="GPKG", mode="a"
            )

        total_registros += len(muni_equip)
        print(f"  → {len(muni_equip)} adicionados. Total até agora: {total_registros}")

    except Exception as e:
        msg = f"{nm_mun} ({cd_mun}) → {e}"
        print(f"⚠ {msg}")
        erros.append(msg)
        continue

# ----------------------------
# SALVAR LISTA DE ERROS
# ----------------------------
if erros:
    with open("./data/erros_municipios.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(erros))
    print("⚠ Lista de municípios com erro salva em ./data/erros_municipios.txt")

# ----------------------------
# RELATÓRIO FINAL
# ----------------------------
if primeiro:
    print("Nenhum município gerou dados. GPKG não criado.")
else:
    print("✅ GPKG salvo em:", OUT_GPKG)
    print("Total final de registros:", total_registros)


In [ ]:
import os
import geopandas as gpd
import pandas as pd
import osmnx as ox

# ----------------------------
# CONFIGURAÇÕES BÁSICAS
# ----------------------------
OUTDIR = "./data/green_jundiai"
os.makedirs(OUTDIR, exist_ok=True)

# Arquivo com limites municipais
SHAPE_MUNICIPIOS = r"D:\Arq-Azzoni\UrbanSprawl\Bases_dados\Shapes\BR_Municipios_2024\BR_Municipios_2024.shp"

# Nome do município a filtrar
MUN_NOME = "Jundiaí"


# 1) Carregar município do shapefile externo
limites = gpd.read_file(SHAPE_MUNICIPIOS)
muni = limites[limites["NM_MUN"] == MUN_NOME].copy()
muni = muni.to_crs(4326)

CD_MUN = muni["CD_MUN"].iloc[0]
NM_MUN = muni["NM_MUN"].iloc[0]

# 2) Baixar dados OSM
place = f"{MUN_NOME}, São Paulo, Brasil"

tags_parques = {"leisure": ["park", "garden"], "landuse": ["recreation_ground"]}
parks = ox.features_from_place(place, tags_parques)

tags_pracas = {"place": "square"}
squares = ox.features_from_place(place, tags_pracas)

tags_ped = {"highway": "pedestrian", "area": "yes"}
ped_areas = ox.features_from_place(place, tags_ped)
ped_areas = ped_areas[ped_areas.get("name", "").astype(str).str.startswith("Praça")]

# 3) Função de recorte
def clean_clip(gdf):
    if gdf.empty:
        return gdf
    gdf = gdf.set_crs(4326, allow_override=True)
    gdf = gpd.clip(gdf, muni)
    return gdf[gdf.geometry.type.isin(["Polygon", "MultiPolygon"])]

parks_poly = clean_clip(parks)
squares_poly = clean_clip(squares)
ped_pracas_poly = clean_clip(ped_areas)

# 4) Unir praças
pracas_poly = gpd.GeoDataFrame(
    pd.concat([squares_poly, ped_pracas_poly], ignore_index=True),
    crs=4326
)

# 5) Criar tabela final com colunas específicas
# Seleciona apenas geometry, name e leisure se existirem
parks_out = parks_poly[["geometry", "name", "leisure"]].copy()
parks_out["equipamento"] = "parque"

pracas_out = pracas_poly[["geometry", "name", "leisure"]].copy()
pracas_out["equipamento"] = "praça"

# Unir parques e praças
equipamentos = gpd.GeoDataFrame(
    pd.concat([parks_out, pracas_out], ignore_index=True),
    crs=4326
)

# Adicionar CD_MUN e NM_MUN
equipamentos["CD_MUN"] = CD_MUN
equipamentos["NM_MUN"] = NM_MUN

# 6) Salvar saída
equipamentos.to_file(os.path.join(OUTDIR, "equipamentos_verdes_jundiai.shp"))
muni.to_file(os.path.join(OUTDIR, "limite_municipal_jundiai.shp"))

print("✅ Shapefile final salvo em:", OUTDIR)

In [ ]:
l = gpd.read_file(r"D:\Arq-Azzoni\UrbanSprawl\Bases_dados\Shapes\BR_Municipios_2024\BR_Municipios_2024.shp")

In [ ]:
l

In [ ]:
import os
import geopandas as gpd
import pandas as pd
import osmnx as ox


OUTDIR = "./data"
os.makedirs(OUTDIR, exist_ok=True)

# 1) Limite municipal de Jundiaí
place = "Jundiaí, São Paulo, Brasil"
muni = ox.geocode_to_gdf(place)            # boundary em WGS84 (EPSG:4326)

# 2) Parques e jardins (leisure=park|garden, + áreas de recreação)
tags_parques = {"leisure": ["park", "garden"], "landuse": ["recreation_ground"]}
parks  = ox.features_from_place(place, tags_parques)

# 3) Praças
tags_pracas = {"place": "square"}
squares  = ox.features_from_place(place, tags_pracas)

# 3b) Áreas pedestrian com nome "Praça ..."
tags_poligonos_diversos = {"highway": "pedestrian", "area": "yes"}
ped_areas = ox.features_from_place(place, tags_poligonos_diversos)
ped_areas = ped_areas[ped_areas.get("name", "").astype(str).str.startswith("Praça")]

# 4) Padronizar CRS e recortar pelo limite
def clean_clip(gdf):
    if gdf.empty:
        return gdf
    gdf = gdf.set_crs(4326, allow_override=True)
    gdf = gpd.clip(gdf, muni.to_crs(4326))
    return gdf[gdf.geometry.type.isin(["Polygon", "MultiPolygon"])]

parks_poly       = clean_clip(parks)
squares_poly     = clean_clip(squares)
ped_pracas_poly  = clean_clip(ped_areas)

# 5) Unir as praças de múltiplas fontes
pracas_poly = gpd.GeoDataFrame(
    pd.concat([squares_poly, ped_pracas_poly], ignore_index=True),
    crs=4326
)

# 6) Criar uma única camada com coluna "equipamento"
#    (mantendo só geometry + equipamento para evitar conflito de campos)
parks_out = parks_poly[["geometry"]].copy()
parks_out["equipamento"] = "parque"

pracas_out = pracas_poly[["geometry"]].copy()
pracas_out["equipamento"] = "praça"

equipamentos = gpd.GeoDataFrame(
    pd.concat([parks_out, pracas_out], ignore_index=True),
    crs=4326
)

# 7) Salvar shapefile único
equipamentos.to_file(os.path.join(OUTDIR, "equipamentos_verdes_jundiai.shp"))

# (opcional) ainda salvar limite municipal, se quiser
#muni.to_file(os.path.join(OUTDIR, "limite_municipal_jundiai.shp"))

print("✅ Shapefile único salvo em:", OUTDIR)


In [ ]:
import os
import geopandas as gpd
import pandas as pd
import osmnx as ox


OUTDIR = "./data/green_jundiai"
os.makedirs(OUTDIR, exist_ok=True)

# 1) Limite municipal de Jundiaí
place = "Jundiaí, São Paulo, Brasil"
muni = ox.geocode_to_gdf(place)            # boundary em WGS84 (EPSG:4326)

# 2) Parques e jardins (leisure=park|garden, + áreas de recreação)
tags_parques = {"leisure": ["park", "garden"], "landuse": ["recreation_ground"]}
parks  = ox.features_from_place(place, tags_parques)

# 3) Praças
# - Em OSM, “praça” aparece principalmente como:
#   a) place=square (ponto ou polígono)
#   b) áreas com nome iniciando por “Praça ...” (mesmo que tag não seja place=square)
tags_pracas = {"place": "square"}
squares  = ox.features_from_place(place, tags_pracas)


# 3b) Capturar polígonos com nome "Praça ..." mesmo sem place=square
#     (ex.: áreas pedestrian, highway=pedestrian area, etc.)
tags_poligonos_diversos = {"highway": "pedestrian", "area": "yes"}
ped_areas = ox.features_from_place(place, tags_poligonos_diversos)
ped_areas = ped_areas[ped_areas.get("name", "").astype(str).str.startswith("Praça")]

# 4) Padronizar CRS e recortar pelo limite (só para garantir)
def clean_clip(gdf):
    if gdf.empty:
        return gdf
    gdf = gdf.set_crs(4326, allow_override=True)
    gdf = gpd.clip(gdf, muni.to_crs(4326))
    # manter só geometrias de área p/ shapefile de polígonos
    return gdf[gdf.geometry.type.isin(["Polygon", "MultiPolygon"])]

parks_poly  = clean_clip(parks)
squares_poly = clean_clip(squares)
ped_pracas_poly = clean_clip(ped_areas)

# 5) Unir as praças de múltiplas fontes
pracas_poly = gpd.GeoDataFrame(pd.concat([squares_poly, ped_pracas_poly], ignore_index=True), crs=4326)

# 6) Salvar Shapefiles
parks_poly.to_file(os.path.join(OUTDIR, "parques_jundiai.shp"))
pracas_poly.to_file(os.path.join(OUTDIR, "pracas_jundiai.shp"))
muni.to_file(os.path.join(OUTDIR, "limite_municipal_jundiai.shp"))

print("✅ Salvo em:", OUTDIR)

In [ ]:
import geopandas as gpd
import folium


# reproject to WGS84 (lat/lon) if needed
if parques.crs and parques.crs.to_string() != "EPSG:4326":
   parques = parques.to_crs(epsg=4326)

   # reproject to WGS84 (lat/lon) if needed
if pracas.crs and pracas.crs.to_string() != "EPSG:4326":
   pracas = pracas.to_crs(epsg=4326)

# --- base map ---
m = folium.Map(location=[-15, -55], zoom_start=5, tiles="CartoDB positron")
# --- add shapefile as a layer ---
folium.GeoJson(
    data=parques,
    name="Parques",
    style_function=lambda f: {
        "color": "darkgreen",
        "weight": 2,
        "fillColor": "darkgreen",
        "fillOpacity": 0.3,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['name'],  # show first two attrs in tooltip
        aliases=["Nome"]
    )
).add_to(m)

folium.GeoJson(
    data=pracas,
    name="Praças",
    style_function=lambda f: {
        "color": "green",
        "weight": 2,
        "fillColor": "green",
        "fillOpacity": 0.3,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['name'],  # show first two attrs in tooltip
        aliases=["Nome"]
    )
).add_to(m)

# --- layer control + save ---
folium.LayerControl().add_to(m)

In [ ]:
m.save("map_green_ju.html")

In [ ]:
parques["type"] = parques["name"].str.split().str[0]

In [ ]:
import re

# garantir que a coluna seja string
parques["type"] = parques["type"].astype(str)

# 1. Remover aspas tipográficas antes de "Praça" ou "PRAÇA"
parques["type"] = parques["type"].str.replace(r'^[“”]\s*(?i:praça)', "Praça", regex=True)

# 2. Substituir "PRAÇA" no início por "Praça"
parques["type"] = parques["type"].str.replace(r'^PRAÇA', "Praça", regex=True)

In [ ]:
parques["type"] = parques["type"].str.replace('Park', "Parque", regex=True)

In [ ]:
parques["type"] = parques["type"].str.replace('"Praça', "Praça", regex=True)

In [ ]:
excluir = ["None", "Renato", "Jardim"]

# manter apenas os que NÃO estão na lista
parques = parques[~parques["type"].isin(excluir)].copy()

In [ ]:
mapa = {"Parque": "Parque", "Praça": "Praça"}
parques["category"] = parques["type"].map(mapa).fillna("Outros")

In [ ]:
import os
import re
import geopandas as gpd

# gdf base (ex.: 'parques') já deve ter a coluna 'category' criada antes
# valores esperados: "Parque", "Praça", "Outros"
assert "category" in parques.columns, "Crie a coluna 'category' antes de exportar."

OUTDIR = "./data/green_jundiai/by_category"
os.makedirs(OUTDIR, exist_ok=True)

def slug(s: str) -> str:
    """Nome de arquivo seguro (sem espaços/acentos problemáticos)."""
    s = str(s).strip()
    s = re.sub(r"[^\w\-]+", "_", s, flags=re.UNICODE)
    s = re.sub(r"_+", "_", s)
    return s.strip("_") or "categoria"

# (opcional) garanta CRS definido
if parques.crs is None:
    parques = parques.set_crs(4326)

# exportar um .shp por categoria
for cat, gdf_cat in parques.groupby("category"):
    if gdf_cat.empty:
        continue
    fname = f"{slug(cat)}.shp"         # ex.: Parque.shp, Praça.shp, Outros.shp
    outpath = os.path.join(OUTDIR, fname)
    gdf_cat.to_file(outpath, driver="ESRI Shapefile", encoding="utf-8")
    print(f"✅ {cat}: {len(gdf_cat)} feições → {outpath}")

In [ ]:
import geopandas as gpd

In [ ]:
parque = gpd.read_file("./data/green_jundiai/by_category\Parque.shp").to_crs(4326)
pracas = gpd.read_file("./data/green_jundiai/by_category\Praça.shp").to_crs(4326)
outros = gpd.read_file("./data/green_jundiai/by_category\Outros.shp").to_crs(4326)

In [ ]:
parque = gpd.read_file("./data/green_jundiai/parques_jundiai.shp").to_crs(4326)
pracas = gpd.read_file("./data/green_jundiai/pracas_jundiai.shp").to_crs(4326)


In [ ]:
pracas

In [ ]:
import geopandas as gpd
import folium


# reproject to WGS84 (lat/lon) if needed
if parque.crs and parque.crs.to_string() != "EPSG:4326":
   parque = parque.to_crs(epsg=4326)

   # reproject to WGS84 (lat/lon) if needed
if pracas.crs and pracas.crs.to_string() != "EPSG:4326":
   pracas = pracas.to_crs(epsg=4326)

      # reproject to WGS84 (lat/lon) if needed
if outros.crs and outros.crs.to_string() != "EPSG:4326":
   outros = outros.to_crs(epsg=4326)

# --- base map ---
m = folium.Map(location=[-15, -55], zoom_start=5, tiles="CartoDB positron")
# --- add shapefile as a layer ---
folium.GeoJson(
    data=parque,
    name="Parque",
    style_function=lambda f: {
        "color": "darkgreen",
        "weight": 2,
        "fillColor": "darkgreen",
        "fillOpacity": 0.3,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['name'],  # show first two attrs in tooltip
        aliases=["Nome"]
    )
).add_to(m)

folium.GeoJson(
    data=pracas,
    name="Praças",
    style_function=lambda f: {
        "color": "lightgreen",
        "weight": 2,
        "fillColor": "lightgreen",
        "fillOpacity": 0.3,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['name'],  # show first two attrs in tooltip
        aliases=["Nome"]
    )
).add_to(m)

folium.GeoJson(
    data=outros,
    name="Praças",
    style_function=lambda f: {
        "color": "blue",
        "weight": 2,
        "fillColor": "lightblue",
        "fillOpacity": 0.3,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['name'],  # show first two attrs in tooltip
        aliases=["Nome"]
    )
).add_to(m)


# --- layer control + save ---
folium.LayerControl().add_to(m)

In [ ]:
m.save("map_green_ju_category.html")